In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.3 Positive Definiteness, Quadratic Forms, and Cholesky

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume III — Eigenvalues and Spectral Theory",
    number="3.3",
    title="Positive Definiteness, Quadratic Forms, and Cholesky",
    blurb="Five conditions that are the same condition, the factorization that "
    "is twice as fast as LU and needs no pivoting, and the reason every "
    "covariance matrix and every kernel in the rest of this course is "
    "guaranteed to behave.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

[§3.2](spectral-theorem.ipynb) gave symmetric matrices real eigenvalues.
Requiring those eigenvalues to be *positive* is a small-sounding extra
condition with disproportionate consequences, and this notebook is about how
many apparently unrelated statements turn out to be that one condition in
disguise.

A symmetric $A$ is **positive definite** when $\mathbf{x}^{\top}\!A\mathbf{x} > 0$
for every $\mathbf{x} \ne \mathbf{0}$. Equivalently: all its eigenvalues are
positive; all its leading principal minors are positive; it has a Cholesky
factorization $A = LL^{\top}$; and it is $G^{\top}G$ for some $G$ with
independent columns. Five tests, one property — and they are *not* equally
useful in floating point, which is a distinction the notebook makes carefully.

The geometry is the reason the definition is natural. The level set
$\mathbf{x}^{\top}\!A\mathbf{x} = 1$ is an **ellipse** exactly when $A$ is
positive definite, with semi-axes $1/\sqrt{\lambda_i}$ along the eigenvectors;
drop definiteness and it becomes a hyperbola or a pair of lines. So a positive
definite matrix is one that defines a sensible notion of squared length, which
is what makes it the right object for an energy, a covariance, or a kernel.

The algorithm is **Cholesky**, and it is the best-behaved factorization in this
course: half the cost of $LU$, no pivoting needed, no growth factor to worry
about, and — usefully — it *fails* precisely when the matrix is not positive
definite, which turns a factorization into a test. It also generates
correlated random vectors in one line, which is how every Gaussian sample in
the rest of this course is drawn.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Strang {cite}`strang2023` Chapter 6; Golub and Van Loan
> {cite}`golub2013` §4.2 for Cholesky and its error analysis; Horn and Johnson
> {cite}`horn2013` Chapter 7 for the equivalences and Sylvester's law; Higham
> {cite}`higham2002` Chapter 10 on why no pivoting is needed.

## Theory in brief

### The definition

A symmetric $A \in \mathbb{R}^{n\times n}$ is **positive definite** if

```{math}
:label: eq-pd-definition
\mathbf{x}^{\top}\!A\mathbf{x} > 0 \quad\text{for every } \mathbf{x} \ne \mathbf{0},
```

**positive semidefinite** if $\ge 0$ is allowed, **negative definite** if the
inequality reverses, and **indefinite** if
$\mathbf{x}^{\top}\!A\mathbf{x}$ takes both signs. The scalar
$\mathbf{x}^{\top}\!A\mathbf{x}$ is the **quadratic form**, and by
[§3.2](spectral-theorem.ipynb) it equals $\sum\lambda_ic_i^2$ in eigenvector
coordinates — so its sign is decided entirely by the signs of the eigenvalues.

### Five equivalent tests

For symmetric $A$, the following are equivalent:

```{math}
:label: eq-pd-tests
\begin{aligned}
&\text{(i)}   && \mathbf{x}^{\top}\!A\mathbf{x} > 0 \text{ for all } \mathbf{x}\ne\mathbf{0};\\
&\text{(ii)}  && \lambda_i > 0 \text{ for every } i;\\
&\text{(iii)} && \det A_k > 0 \text{ for } k = 1,\dots,n \quad(A_k \text{ the leading } k\times k \text{ block});\\
&\text{(iv)}  && A = LL^{\top} \text{ for a lower triangular } L \text{ with positive diagonal};\\
&\text{(v)}   && A = G^{\top}G \text{ for some } G \text{ with independent columns}.
\end{aligned}
```

Test (iii) is **Sylvester's criterion**, and it is the one that fails most
instructively: it needs *leading* minors, and checking all $2^n - 1$ principal
minors is a different (also valid) criterion, while checking only $\det A > 0$
is no criterion at all — $\operatorname{diag}(-1,-1)$ has determinant $+1$.

### Cholesky

The factorization in (iv) is

```{math}
:label: eq-pd-cholesky
A = LL^{\top},
```

computed by the recurrence, for $j \le i$,

```{math}
:label: eq-pd-cholesky-alg
\ell_{jj} = \sqrt{a_{jj} - \sum_{k<j}\ell_{jk}^2},
\qquad
\ell_{ij} = \frac{1}{\ell_{jj}}\Bigl(a_{ij} - \sum_{k<j}\ell_{ik}\ell_{jk}\Bigr) .
```

It costs $n^3/3$ flops, half of $LU$, because symmetry means only one triangle
is touched. It needs **no pivoting**: the growth factor is exactly 1, since
every $\ell_{jk}^2$ in the first formula is subtracted from $a_{jj}$ and the
result must stay positive, which bounds every entry of $L$ by $\sqrt{a_{jj}}$.
That is a guarantee $LU$ cannot make, as [§1.2](../01-matrices/elimination-lu.ipynb)
measured with Wilkinson's matrix. And the square root under the radical goes
negative exactly when $A$ is not positive definite, so the algorithm doubles as
the sharpest of the five tests.

### The energy ellipse

For positive definite $A$, the level set

```{math}
:label: eq-pd-ellipse
\{\mathbf{x} : \mathbf{x}^{\top}\!A\mathbf{x} = 1\}
```

is an ellipsoid whose principal axes point along the eigenvectors
$\mathbf{q}_i$ with semi-axis lengths $1/\sqrt{\lambda_i}$. A *large*
eigenvalue therefore gives a *short* axis: the form grows fastest in the
direction it is stiffest. Indefinite $A$ gives a hyperbola instead, and
semidefinite $A$ gives an unbounded strip, so the shape of the level set is a
complete diagnosis.

### Congruence and Sylvester's law of inertia

For invertible $C$, the map

```{math}
:label: eq-pd-congruence
A \;\longmapsto\; C^{\top}\!AC
```

is a **congruence**. It is what happens to a quadratic form under a change of
variables $\mathbf{x} = C\mathbf{y}$, and unlike a similarity transformation it
does *not* preserve the eigenvalues. What it does preserve is

```{math}
:label: eq-pd-inertia
\operatorname{In}(A) = (n_+,\, n_0,\, n_-),
```

the counts of positive, zero and negative eigenvalues — **Sylvester's law of
inertia**. So positive definiteness is a property of the quadratic form itself
and not of the coordinates it is written in, which is exactly why it is worth
having a name.

### Gram matrices, covariance, and sampling

For any $G$,

```{math}
:label: eq-pd-gram
\mathbf{x}^{\top}(G^{\top}G)\mathbf{x} = \|G\mathbf{x}\|_2^2 \ge 0 ,
```

so $G^{\top}G$ is always positive semidefinite, and positive *definite* exactly
when $G$ has independent columns. That single line explains why every Gram
matrix, every covariance matrix and every kernel matrix in this course is
guaranteed to be at least semidefinite.

Run the implication backwards and Cholesky becomes a sampler. If
$\mathbf{z}$ has independent standard normal entries and $\Sigma = LL^{\top}$,
then

```{math}
:label: eq-pd-sampling
\mathbf{x} = L\mathbf{z}
\quad\Longrightarrow\quad
\operatorname{Cov}(\mathbf{x}) = L\operatorname{Cov}(\mathbf{z})L^{\top} = LL^{\top} = \Sigma .
```

For a 2-D Gaussian the quantity $\mathbf{x}^{\top}\Sigma^{-1}\mathbf{x}$ is
$\chi^2$ with 2 degrees of freedom, so the fraction of samples inside the
$k\sigma$ ellipse has the closed form

```{math}
:label: eq-pd-chi2
\Pr\bigl(\mathbf{x}^{\top}\Sigma^{-1}\mathbf{x} \le k^2\bigr) = 1 - e^{-k^2/2} .
```

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky, cho_factor, cho_solve

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed
# The Monte Carlo in Exercise 6 gets its own generator, so its sample covariance
# does not shift when an earlier exercise changes how many draws it makes.
rng_mc = np.random.default_rng(3)

EPS = np.finfo(float).eps
np.set_printoptions(precision=6, suppress=True, linewidth=120)

# The test suite: four symmetric 3x3 matrices, one of each kind. Every test in
# Exercise 1 is applied to all four, so the rejections carry as much weight as
# the acceptance.
SUITE = {
    "positive definite": np.array([[4.0, 1.0, 1.0],
                                   [1.0, 3.0, -1.0],
                                   [1.0, -1.0, 5.0]]),
    "indefinite": np.array([[1.0, 2.0, 0.0],
                            [2.0, 1.0, 0.0],
                            [0.0, 0.0, 3.0]]),
    "positive semidefinite": np.array([[1.0, 1.0, 0.0],
                                       [1.0, 1.0, 0.0],
                                       [0.0, 0.0, 2.0]]),
    "negative definite": np.array([[-2.0, 1.0, 0.0],
                                   [1.0, -3.0, 1.0],
                                   [0.0, 1.0, -2.0]]),
}
A_PD = SUITE["positive definite"]

# The 2x2 used for every picture, with eigenvalues (5 +/- sqrt 5)/2.
A2 = np.array([[3.0, 1.0],
               [1.0, 2.0]])


def inertia(A, tol=None):
    """The inertia triple (n_+, n_0, n_-) of Eq. 7, from the eigenvalues.

    The tolerance separating "zero" from "small" scales with the matrix, because
    a congruence C^T A C multiplies the eigenvalues by something of order
    ||C||^2 and an absolute threshold would then misclassify them.
    """
    w = np.linalg.eigvalsh(A)
    if tol is None:
        tol = 1e-9 * max(1.0, float(np.abs(w).max()))
    return (int((w > tol).sum()), int((np.abs(w) <= tol).sum()), int((w < -tol).sum()))

## Exercise 1 — Five tests, one property, and two of them unusable

{eq}`eq-pd-tests` lists five conditions that are equivalent *as theorems*. This
exercise applies all five to four matrices — one of each kind — and finds that
two of the tests behave badly enough in floating point that they should not be
used as tests at all.

The suite, all symmetric $3\times3$ and available as `SUITE`:

$$
A_{\text{pd}} = \begin{bmatrix} 4&1&1\\ 1&3&-1\\ 1&-1&5\end{bmatrix},
\quad
A_{\text{ind}} = \begin{bmatrix} 1&2&0\\ 2&1&0\\ 0&0&3\end{bmatrix},
\quad
A_{\text{psd}} = \begin{bmatrix} 1&1&0\\ 1&1&0\\ 0&0&2\end{bmatrix},
\quad
A_{\text{nd}} = \begin{bmatrix} -2&1&0\\ 1&-3&1\\ 0&1&-2\end{bmatrix}.
$$

**Part a)** Apply test (ii). For each matrix compute `np.linalg.eigvalsh` and
report whether every eigenvalue exceeds $10^{-12}$. The spectra are
$\{1.786, 4.539, 5.675\}$, $\{-1, 3, 3\}$, $\{0, 2, 2\}$ and
$\{-4, -2, -1\}$, so only the first passes.

**Part b)** Apply test (iii), Sylvester's criterion, computing the three
leading principal minors `np.linalg.det(A[:k, :k])` for $k = 1, 2, 3$. They
are $(4, 11, 46)$, $(1, -3, -9)$, $(1, 0, 0)$ and $(-2, 5, -8)$. Confirm the
verdict agrees with Part a) on all four, and note the trap: the last matrix
has a *positive* second minor and is negative definite, so no single minor
decides anything.

**Part c)** Apply test (iv). Try `scipy.linalg.cholesky(A, lower=True)` inside
a `try/except` and record whether it succeeded. Confirm it agrees with Parts
a) and b) on all four, raising `LinAlgError` on the three that are not
positive definite — including the *semidefinite* one, where the factorization
exists in exact arithmetic but the algorithm hits a zero pivot.

**Part d)** Now test (i), by sampling, and watch it fail. Draw
$2\times10^{4}$ random unit vectors and report
$\min_{\mathbf{x}}\mathbf{x}^{\top}\!A\mathbf{x}$ for each matrix. For the
semidefinite matrix the sampled minimum is $+1.1\times10^{-4}$ — *positive* —
so sampling declares it positive definite, and it is not. Confirm the sampled
minimum is positive, and confirm that the true minimum, $\lambda_1 = 0$,
is not. A property quantified over all $\mathbf{x}$ cannot be established by
checking finitely many of them, and this is the concrete case.

**Part e)** Apply test (v) where it applies. For the two matrices that are at
least semidefinite, build $G = Q\sqrt{\Lambda}Q^{\top}$ from `eigh` with the
eigenvalues clipped at 0 by `np.clip(w, 0, None)`, and confirm
$G^{\top}G = A$ to $10^{-14}$. Report $\operatorname{rank}(G)$: it is 3 for
the definite matrix and 2 for the semidefinite one, which is test (v)'s
"independent columns" clause doing the work.

**Part f)** Summarise the verdicts in a table and confirm all four *reliable*
tests — eigenvalues, minors, Cholesky, and the rank of $G$ — agree on every
matrix in the suite, while sampling gets one of the four wrong.

In [ ]:
# (solution hidden on the public site)


### Validation 1

The rejections carry as much weight as the acceptance, so all four matrices
are checked by all the reliable tests. The sampling test is reported as
*failing*, deliberately and by a specific amount, because "these five
conditions are equivalent" is a theorem about exact arithmetic and this is
what it costs in practice.

In [ ]:
validate.check(
    reliable_agree,
    "eigenvalues, leading minors and Cholesky agree on all four matrices (Eq. 2)",
    "each test accepts exactly the positive definite matrix and rejects the "
    "indefinite, semidefinite and negative definite ones",
)
validate.check(
    results["negative definite"]["minors"][1] > 0,
    "and no single minor decides: the negative definite matrix has det A_2 > 0",
    f"its leading minors are "
    f"{np.array2string(results['negative definite']['minors'], precision=1)} — "
    "Sylvester's criterion needs ALL of them positive, in order",
)
validate.check(
    sample_wrong == ["positive semidefinite"],
    "while sampling the quadratic form calls the semidefinite matrix definite",
    f"minimum over 20,000 random unit vectors is "
    f"{results['positive semidefinite']['q_min']:+.3e} > 0, against a true "
    f"minimum of {results['positive semidefinite']['w'][0]:+.1e}: a property "
    "quantified over all x is not decidable by finitely many samples",
)
validate.check(
    gram_info["positive definite"][0] < 1e-14
    and gram_info["positive semidefinite"][0] < 1e-14,
    "both semidefinite matrices factor as G^T G (Eq. 8)",
    f"reconstruction errors {gram_info['positive definite'][0]:.2e} and "
    f"{gram_info['positive semidefinite'][0]:.2e}",
)
validate.check(
    gram_info["positive definite"][1] == 3
    and gram_info["positive semidefinite"][1] == 2,
    "and rank(G) separates them: 3 independent columns against 2",
    "test (v) asks for INDEPENDENT columns, and that clause is exactly what "
    "distinguishes definite from semidefinite",
)

## Exercise 2 — Cholesky, and why it needs no pivoting

{eq}`eq-pd-cholesky-alg` is the algorithm, and it is unusually well behaved.
[§1.2](../01-matrices/elimination-lu.ipynb) showed that Gaussian elimination
without pivoting can fail outright and with pivoting can still suffer a growth
factor of $2^{n-1}$. Cholesky has a growth factor of exactly 1, needs no row
interchanges, costs half as much, and — because the square root argument goes
negative precisely when the matrix is not positive definite — reports failure
rather than returning nonsense.

**Part a)** Write `cholesky_hand(A)` implementing {eq}`eq-pd-cholesky-alg` and
returning the lower triangular $L$. Raise `ValueError("not positive definite")`
if any diagonal argument is $\le 0$. Loop over $i$ from 0 to $n-1$ and $j$ from
0 to $i$, forming `s = A[i, j] - L[i, :j] @ L[j, :j]`, then setting
`L[i, i] = np.sqrt(s)` on the diagonal and `L[i, j] = s / L[j, j]` below it.

**Write this one yourself** — the implementation is the lesson.

**Part b)** Run it on $A_{\text{pd}}$ and confirm $LL^{\top} = A$ to
$10^{-14}$, that $L$ is *exactly* lower triangular (`np.triu(L, 1)` is
identically zero, not merely small), and that it agrees with
`scipy.linalg.cholesky(A_PD, lower=True)` to $10^{-14}$.

**Part c)** Confirm the failure mode. Call `cholesky_hand` on the indefinite
and the semidefinite matrices and confirm both raise. The semidefinite case is
the interesting one: $A = LL^{\top}$ does exist for it in exact arithmetic
(with a zero on $L$'s diagonal), but the *algorithm* divides by that zero, so
the factorization is unavailable even though it exists.

**Part d)** Confirm the growth bound. For $A_{\text{pd}}$, check that every
entry of $L$ satisfies $|\ell_{ij}| \le \sqrt{a_{jj}}$, which follows from
$\ell_{jj}^2 + \sum_{k<j}\ell_{jk}^2 = a_{jj}$: each row of $L$ has squared
norm equal to a diagonal entry of $A$. Verify that identity directly —
$\|L_{i,:}\|^2 = a_{ii}$ for every $i$, to $10^{-14}$ — since it *is* the
statement that no entry can grow.

**Part e)** Use it as a solver. Solve $A_{\text{pd}}\mathbf{x} = \mathbf{b}$
for `b = np.array([1.0, 2.0, 3.0])` two ways: with
`scipy.linalg.cho_factor` followed by `cho_solve`, and with
`np.linalg.solve`. Confirm they agree to $10^{-13}$ and that the residual
$\|A\mathbf{x} - \mathbf{b}\|$ is below $10^{-14}$.

**Part f)** State the cost from the model and then measure it. Report
`la.flops("lu", 800) / la.flops("cholesky", 800)`, which is exactly 2 —
$2n^3/3$ against $n^3/3$ — and gate *that*. Then, for $n = 200, 400, 800$,
build `la.random_spd(n, rng)` and time `scipy.linalg.cholesky` against
`scipy.linalg.lu_factor`, and **report** the ratios without gating them.

The distinction is the course's rule and it is worth being strict about. A
stated FLOP count is a property of the algorithm; a wall-clock ratio is a
property of the machine that ran it. At these sizes both routines move more
memory than they do arithmetic, so the factor of 2 is only partly realised —
and on a shared CI runner one of these three ratios has come back at $0.37$,
with Cholesky apparently *slower*, on exactly the code that gives $1.9$ here.
Gating that number at any threshold would be testing the runner's load.

In [ ]:
# (solution hidden on the public site)


### Validation 2

The identity $\|L_{i,:}\|^2 = a_{ii}$ is the check worth having: it is exact,
it is the reason no pivoting is needed, and it makes "the growth factor is 1"
into something measurable rather than something asserted. The timing is gated
only as an inequality, never as a ratio, since machines differ.

In [ ]:
validate.close(
    L @ L.T, A_PD,
    "the hand-written Cholesky reconstructs A = L L^T (Eqs. 3, 4)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    upper_zero == 0.0 and scipy_gap < 1e-14,
    "with L exactly lower triangular and matching scipy.linalg.cholesky",
    f"strictly upper triangle is identically {upper_zero:.0f}; agreement with "
    f"scipy {scipy_gap:.2e}",
)
validate.check(
    all(v is not None for v in failures.values()),
    "and it raises on all three non-positive-definite matrices (Eq. 2 (iv))",
    "including the semidefinite one, where A = L L^T exists in exact arithmetic "
    "with a zero on the diagonal but the algorithm must divide by it",
)
validate.close(
    row_norms, np.diag(A_PD),
    "each row of L has squared norm a_ii: the growth factor is exactly 1",
    rtol=0.0, atol=1e-14,
)
validate.check(
    np.abs(x_chol - x_solve).max() < 1e-13
    and np.linalg.norm(A_PD @ x_chol - b) < 1e-14,
    "cho_factor/cho_solve agrees with np.linalg.solve on A x = b",
    f"difference {np.abs(x_chol - x_solve).max():.2e}, residual "
    f"{np.linalg.norm(A_PD @ x_chol - b):.2e}",
)
validate.close(
    np.array([flop_ratio]), np.array([2.0]),
    "Cholesky costs n^3/3 against LU's 2n^3/3: exactly half, from the cost model",
    rtol=0.0, atol=1e-12,
)
print(f"\nmeasured lu/cholesky time ratios: "
      f"{[f'{r:.2f}' for r in timing_ratios]} at n = 200, 400, 800, averaging "
      f"{np.mean(timing_ratios):.2f} against the predicted 2.00. This is "
      f"REPORTED and not gated: a wall-clock ratio on a shared runner is not a "
      f"property of the algorithms. A CI machine has measured 0.37 at n = 800 "
      f"on the same code that gives 1.9 here.")

## Exercise 3 — The energy ellipse, and what the axes measure

{eq}`eq-pd-ellipse` is where the definition becomes geometry. For positive
definite $A$ the level set $\mathbf{x}^{\top}\!A\mathbf{x} = 1$ is a closed
ellipse; the moment an eigenvalue turns negative it opens into a hyperbola, and
when one hits zero it degenerates into a strip. The shape *is* the
classification.

The axis lengths are worth getting the right way round. In eigenvector
coordinates the form is $\sum\lambda_iy_i^2 = 1$, so the ellipse reaches
$y_i = 1/\sqrt{\lambda_i}$ along $\mathbf{q}_i$: **large eigenvalue, short
axis**. The matrix is stiffest in the direction where the form grows fastest,
which is the direction the ellipse is thinnest.

The matrix is $A_2 = \left[\begin{smallmatrix}3&1\\1&2\end{smallmatrix}\right]$,
available as `A2`, with eigenvalues $(5 \pm \sqrt5)/2 = 1.381966$ and
$3.618034$.

**Part a)** Confirm the eigenvalues of `A2` are $(5-\sqrt5)/2$ and
$(5+\sqrt5)/2$ to $10^{-14}$, and report the predicted semi-axis lengths
$1/\sqrt{\lambda_i}$, which are $0.850651$ and $0.525731$.

**Part b)** Trace the ellipse numerically. For $200{,}001$ angles $\theta$ in
$[0, 2\pi]$ take the unit vector $\mathbf{u}(\theta)$ and scale it to
$\mathbf{x} = \mathbf{u}/\sqrt{\mathbf{u}^{\top}\!A_2\mathbf{u}}$, which lies
on the level set by construction. Confirm
$\mathbf{x}^{\top}\!A_2\mathbf{x} = 1$ along the whole curve to $10^{-14}$.

**Part c)** Measure the semi-axes as $\max\|\mathbf{x}\|$ and
$\min\|\mathbf{x}\|$ over the traced curve, and confirm they equal
$1/\sqrt{\lambda_{\min}}$ and $1/\sqrt{\lambda_{\max}}$ to $10^{-6}$ — limited
by the angular grid, not by the arithmetic. Confirm the *longest* axis goes
with the *smallest* eigenvalue.

**Part d)** Confirm the axis *directions* are the eigenvectors: take the
traced point of maximum norm and check $|\cos\theta|$ against
$\mathbf{q}_{\min}$ is 1 to $10^{-5}$, again grid-limited.

**Part e)** Confirm the classification by shape. For the indefinite
$\left[\begin{smallmatrix}1&2\\2&1\end{smallmatrix}\right]$, with eigenvalues
$-1$ and $3$, check that $\mathbf{u}^{\top}\!A\mathbf{u}$ changes sign around
the circle, so the scaling above is undefined for some $\theta$ and the level
set is unbounded.

The fraction of directions on which it is negative has a closed form. In
eigenvector coordinates the form is $\lambda_-\cos^2\phi + \lambda_+\sin^2\phi$,
which is negative exactly when $\tan^2\phi < |\lambda_-|/\lambda_+$, so the
negative set is four arcs of half-angle $\arctan\sqrt{|\lambda_-|/\lambda_+}$ and

```{math}
:label: eq-pd-negfrac
\frac{\text{negative directions}}{\text{all directions}}
  = \frac{2}{\pi}\arctan\sqrt{\frac{|\lambda_-|}{\lambda_+}} .
```

Here $|\lambda_-|/\lambda_+ = 1/3$ and $\arctan(1/\sqrt3) = \pi/6$, so the
fraction is exactly $1/3$ — *not* one half, which is worth noticing: an
indefinite matrix need not be negative in half its directions, and how much of
the circle it loses is set by the ratio of the eigenvalues. Confirm the
measured fraction against {eq}`eq-pd-negfrac` to $10^{-4}$.

**Part f)** Draw the contours of the quadratic form for the definite, the
indefinite and the semidefinite $2\times2$ cases side by side, with the level
set $\mathbf{x}^{\top}\!A\mathbf{x} = 1$ picked out, so the ellipse, the
hyperbola and the pair of lines can be compared.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

The semi-axes are checked against $1/\sqrt{\lambda_i}$ to $10^{-6}$ rather than
machine precision, and the reason is stated: the curve is traced on a finite
angular grid, so the measurement is grid-limited. Gating it tighter would be
testing the grid spacing.

In [ ]:
validate.close(
    lam2, exact_lam2,
    "A_2 has eigenvalues (5 -/+ sqrt 5)/2 (Eq. 2)",
    rtol=0.0, atol=1e-14,
)
validate.check(
    on_level < 1e-14,
    "the traced curve lies on the level set x^T A x = 1 (Eq. 5)",
    f"largest deviation from 1 over 200,001 points is {on_level:.2e}",
)
validate.close(
    measured, np.array([predicted_axes[0], predicted_axes[1]]),
    "and its semi-axes are 1/sqrt(lambda_i), grid-limited to 1e-6 (Eq. 5)",
    rtol=0.0, atol=1e-6,
)
validate.check(
    measured[0] > measured[1] and lam2[0] < lam2[1],
    "with the LONGEST axis belonging to the SMALLEST eigenvalue",
    f"semi-axis {measured[0]:.6f} for lambda = {lam2[0]:.6f} against "
    f"{measured[1]:.6f} for lambda = {lam2[1]:.6f}: the form grows fastest where "
    "the ellipse is thinnest",
)
validate.close(
    np.array([axis_cos]), np.array([1.0]),
    "and the long axis points along the eigenvector, to grid resolution",
    rtol=0.0, atol=1e-5,
)
validate.close(
    np.array([neg_frac]), np.array([neg_exact]),
    "and the indefinite form is negative on exactly 1/3 of directions (Eq. 11)",
    rtol=0.0, atol=1e-4,
)
validate.close(
    np.array([neg_exact]), np.array([1.0 / 3.0]),
    "which the closed form gives as 2 arctan(1/sqrt 3)/pi = 1/3, not 1/2",
    rtol=0.0, atol=1e-15,
)

## Exercise 4 — Congruence, and Sylvester's law of inertia

A change of variables $\mathbf{x} = C\mathbf{y}$ turns the quadratic form
$\mathbf{x}^{\top}\!A\mathbf{x}$ into
$\mathbf{y}^{\top}(C^{\top}\!AC)\mathbf{y}$, so {eq}`eq-pd-congruence` is what
a coordinate change does to a symmetric matrix. It is *not* a similarity
transformation: $C^{\top} \ne C^{-1}$ in general, and the eigenvalues move
freely.

What survives is {eq}`eq-pd-inertia`. The number of positive, zero and negative
eigenvalues is invariant under every congruence, which is Sylvester's law of
inertia and is the reason "positive definite" is a property of the form rather
than of the matrix that happens to represent it in some basis.

**Part a)** Confirm eigenvalues are *not* preserved. Take $A_{\text{pd}}$ and
one random invertible $C$, form $C^{\top}\!A_{\text{pd}}C$, and report both
spectra. They differ substantially — report the ratio of the largest
eigenvalues — while both are entirely positive.

**Part b)** Confirm the *inertia* is preserved, for all four matrices in the
suite. For each, apply 100 random congruences with
`C = rng.standard_normal((3, 3))` rejected and redrawn while
$|\det C| < 0.1$ (so $C$ is safely invertible), and confirm `inertia` returns
the same triple every time. The four triples are $(3,0,0)$, $(2,0,1)$,
$(2,1,0)$ and $(0,0,3)$.

**Part c)** Note why `inertia` uses a *scaled* tolerance. A congruence
multiplies the eigenvalues by something of order $\|C\|^2$, so the boundary
between "zero" and "small" moves with $C$; an absolute threshold would
misclassify the semidefinite matrix's zero eigenvalue. Confirm this matters by
reporting, for the semidefinite matrix, the largest and smallest $|\lambda|$
over the 100 congruences — they span several orders of magnitude while the
triple never changes.

**Part d)** Confirm the special case that makes Cholesky an instance of the
law: $A = LL^{\top}$ is the congruence $C^{\top}IC$ with $C = L^{\top}$, so a
positive definite matrix is congruent to the identity. Verify directly that
$L^{-1}A_{\text{pd}}L^{-\top} = I$ to $10^{-14}$, using
`scipy.linalg.solve_triangular` rather than forming the inverse.

**Part e)** Confirm the converse direction of the law is what does the work in
practice: two symmetric matrices are congruent **if and only if** they have the
same inertia. Check the "only if" half constructively for $A_{\text{ind}}$,
whose inertia is $(2,0,1)$: build $D = \operatorname{diag}(1, 1, -1)$, which
has the same inertia, and confirm an explicit congruence connects them, by
forming $C = Q\Lambda^{-1/2}_{\text{abs}}$ from `eigh` and checking
$C^{\top}A_{\text{ind}}C$ equals a diagonal matrix of $\pm1$ to $10^{-13}$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

The law is checked as an *exact integer* statement — a triple of counts either
matches or does not — across 400 congruences, which is the right shape of check
for a combinatorial invariant. The scaled tolerance inside `inertia` is
justified by measurement rather than by assertion.

In [ ]:
validate.check(
    np.abs(np.linalg.eigvalsh(cong0) - np.linalg.eigvalsh(A_PD)).max() > 1.0,
    "a congruence does NOT preserve the eigenvalues (Eq. 6)",
    f"{np.array2string(np.linalg.eigvalsh(A_PD), precision=3)} becomes "
    f"{np.array2string(np.linalg.eigvalsh(cong0), precision=3)}: C^T != C^-1, so "
    "this is not a similarity transformation",
)
validate.check(
    inertia_ok,
    "but it preserves the inertia, over 400 congruences of four matrices (Eq. 7)",
    "triples (3,0,0), (2,0,1), (2,1,0), (0,0,3) unchanged every time — an exact "
    "integer invariant, so there is no tolerance to argue about",
)
validate.check(
    psd_span[1] / psd_span[0] > 100.0,
    "and the zero test must scale, because the nonzero eigenvalues move freely",
    f"for the semidefinite matrix they ranged over {psd_span[0]:.2e} to "
    f"{psd_span[1]:.2e}, a span of {psd_span[1]/psd_span[0]:.0f}x, while the "
    "zero eigenvalue stayed zero",
)
validate.close(
    cong_I, np.eye(3),
    "Cholesky IS a congruence to the identity: L^-1 A L^-T = I (Eqs. 3, 6)",
    rtol=0.0, atol=1e-14,
)
validate.close(
    D_signs, np.diag(np.sign(w_i)),
    "and every symmetric matrix is congruent to a diagonal matrix of signs",
    rtol=0.0, atol=1e-13,
)

## Exercise 5 — Every Gram matrix is semidefinite, and when it is definite

{eq}`eq-pd-gram` is one line long and it underwrites a great deal of what
follows in this course. $\mathbf{x}^{\top}(G^{\top}G)\mathbf{x} =
\|G\mathbf{x}\|^2 \ge 0$ always, with equality exactly when
$G\mathbf{x} = \mathbf{0}$ — so $G^{\top}G$ is positive semidefinite for
*every* $G$, and positive definite precisely when $G$ has independent columns,
that is when $G$'s null space is trivial.

This is why $A^{\top}A$ appeared as a positive definite matrix throughout
[§2.1](../02-orthogonality/projections-normal-equations.ipynb) and
[§2.3](../02-orthogonality/least-squares-four-ways.ipynb) without ever being
argued for, and it is the reason every covariance and kernel matrix in Volumes
IV and VIII is guaranteed to behave.

**Part a)** For the four shapes $(m, n) = (10, 4), (4, 10), (50, 50), (3, 8)$,
draw `G = rng.standard_normal((m, n))` and form $M = G^{\top}G$. Report
$\lambda_{\min}(M)$ for each.

**Part b)** Confirm semidefiniteness for all four: $\lambda_{\min}$ scaled by
$\|M\|_2$ is at least $-10^{-14}$. It comes out around $-10^{-16}$ for the
rank-deficient cases, which is rounding on an exact zero, and this is the
reason such a check must be scaled and one-sided rather than demanding
$\lambda_{\min} \ge 0$ outright.

**Part c)** Confirm the definiteness criterion. The tall cases $(10, 4)$ and
$(50, 50)$ have independent columns and give $\lambda_{\min} > 0$; the wide
cases $(4, 10)$ and $(3, 8)$ cannot, since $n > m$, and give
$\lambda_{\min} \approx 0$. Confirm this pattern, and confirm
$\operatorname{rank}(G^{\top}G) = \operatorname{rank}(G)$ in every case, which
is the statement of [§1.3](../01-matrices/inverses-rank-cr.ipynb) reappearing.

**Part d)** Confirm the identity {eq}`eq-pd-gram` numerically rather than
taking it on faith: for the $(10, 4)$ case and $10^{4}$ random $\mathbf{x}$,
check $\mathbf{x}^{\top}(G^{\top}G)\mathbf{x} = \|G\mathbf{x}\|^2$ to a
relative $10^{-13}$.

**Part e)** Confirm the converse construction, which is test (v) of
{eq}`eq-pd-tests` run backwards: every positive definite matrix *is* a Gram
matrix, and Cholesky exhibits the $G$. For $A_{\text{pd}}$, confirm
$A = (L^{\top})^{\top}L^{\top}$ — that is, $G = L^{\top}$ works — to
$10^{-14}$, and confirm $L^{\top}$ has independent columns by checking its rank
is 3.

```{admonition} With your assistant
:class: tip
A matrix that *should* be positive definite often is not, once it has been
estimated from data or perturbed by rounding: sample covariance matrices from
too few samples come back with small negative eigenvalues, and Cholesky then
refuses. The standard repair is the **nearest positive semidefinite matrix** in
the Frobenius norm, which Higham showed is obtained by taking the symmetric
part, computing `eigh`, clipping the eigenvalues at zero, and reassembling.
Ask your assistant to write `nearest_psd(A)` doing exactly that. Then check it
against the mathematics rather than against a picture: build a deliberately
broken matrix as `A_PD` with its smallest eigenvalue shifted to $-0.05$,
repair it, and verify (i) the result is symmetric to $10^{-14}$, (ii)
`np.linalg.eigvalsh` returns no eigenvalue below $-10^{-14}$, (iii)
`scipy.linalg.cholesky` now succeeds on the repaired matrix plus
$10^{-10}I$, and (iv) the repair changed only the offending eigenvalue — the
other two should be unmoved to $10^{-13}$, since clipping is a projection and
projections do not disturb what already satisfies the constraint. The check is
yours.
```

In [ ]:
# (solution hidden on the public site)


### Validation 5

The semidefiniteness check is **scaled and one-sided**, because $\lambda_{\min}$
of a rank-deficient Gram matrix is an exact zero computed in floating point and
will land on either side of it. Demanding $\lambda_{\min} \ge 0$ would fail on
a correct answer, for the same reason the rank counts in
[§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb) were not
gateable.

In [ ]:
validate.check(
    min(scaled_mins) > -1e-14,
    "G^T G is positive semidefinite for every shape tested (Eq. 8)",
    f"worst scaled lambda_min is {min(scaled_mins):+.2e}; the check is scaled "
    "and one-sided because for a rank-deficient G that eigenvalue is an exact "
    "zero computed in floating point",
)
validate.check(
    [r[4] for r in gram_rows] == [r[0] >= r[1] for r in gram_rows],
    "and positive DEFINITE exactly when G has independent columns (Eq. 8)",
    f"definite for shapes {[f'{r[0]}x{r[1]}' for r in gram_rows if r[4]]} and "
    f"not for {[f'{r[0]}x{r[1]}' for r in gram_rows if not r[4]]}",
)
validate.check(
    all(r[2] == r[3] for r in gram_rows),
    "with rank(G^T G) = rank(G) in every case",
    f"ranks {[(r[2], r[3]) for r in gram_rows]}: the result from 1.3, reappearing",
)
validate.check(
    identity_gap < 1e-13,
    "the identity x^T (G^T G) x = ||G x||^2 holds pointwise (Eq. 8)",
    f"largest relative discrepancy over 10,000 random x is {identity_gap:.2e} — "
    "and that one line is the whole proof of semidefiniteness",
)
validate.close(
    G_from_chol.T @ G_from_chol, A_PD,
    "and conversely every positive definite A is G^T G, with G = L^T (Eq. 3)",
    rtol=0.0, atol=1e-14,
)

## Exercise 6 — Covariance, and generating correlated randomness

{eq}`eq-pd-sampling` turns Cholesky into a sampler. Independent standard
normals have covariance $I$; multiplying by $L$ gives covariance $LL^{\top} =
\Sigma$. Every correlated Gaussian in this course is drawn that way, and it is
the reason a covariance matrix must be positive semidefinite: it is
$\mathbb{E}[\mathbf{x}\mathbf{x}^{\top}]$, a limit of Gram matrices.

The target is
$\Sigma = \left[\begin{smallmatrix}4 & 1.8\\ 1.8 & 1\end{smallmatrix}\right]$,
with correlation $\rho = 1.8/\sqrt{4\cdot1} = 0.9$.

**Part a)** Confirm $\Sigma$ is positive definite by all three reliable tests
of Exercise 1, and compute its Cholesky factor with `np.linalg.cholesky`.

**Part b)** Draw $10^{5}$ samples as `Z = rng_mc.standard_normal((2, 100_000))`
and `X = Lc @ Z`. Report the sample covariance `np.cov(X)` beside $\Sigma$ and
confirm every entry agrees to a relative $4\%$. The measured worst entry is
$1.3\%$; the gate is set at three times that rather than just above it,
because a sample covariance is a random variable and the expected relative
error of an entry from $N$ draws is only about $\sqrt{2/N} = 0.45\%$ — so a
gate placed just past the observed value would be testing the draw, not the
method.

**Part c)** Report the sample correlation against the target $0.9$ and confirm
agreement to $10^{-2}$. Correlation is the scale-free summary and converges a
little faster than the raw covariances.

**Part d)** Confirm the geometry. The set
$\{\mathbf{x} : \mathbf{x}^{\top}\Sigma^{-1}\mathbf{x} = k^2\}$ is the
$k\sigma$ ellipse, and by {eq}`eq-pd-chi2` the fraction of samples inside it is
$1 - e^{-k^2/2}$ exactly. Compute the fraction inside for $k = 1$ and $k = 2$
and compare against $0.393469$ and $0.864665$. Confirm both to $10^{-2}$.
This is a stronger check than the covariance comparison: it tests the whole
distribution, not its second moments.

**Part e)** Confirm the $1\sigma$ ellipse's axes are the eigenvectors of
$\Sigma$ with semi-axes $\sqrt{\lambda_i}$ — note the square root goes the
*other* way here than in Exercise 3, because the ellipse is defined by
$\Sigma^{-1}$ rather than by $\Sigma$. Verify $1/\sqrt{\lambda_i(\Sigma^{-1})}
= \sqrt{\lambda_i(\Sigma)}$ to $10^{-12}$.

**Part f)** Plot $2000$ of the samples with the $1\sigma$ and $2\sigma$
ellipses overlaid and the eigenvector axes drawn, so the correlation is visible
as a tilt.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

The $\chi^2$ fractions are the check that matters. Matching a sample covariance
to a target tests two moments; matching the fraction inside every $k\sigma$
ellipse tests the whole radial distribution, and it has an exact closed form to
compare against, so there is no simulation-versus-simulation comparison
anywhere in it.

In [ ]:
validate.check(
    bool((lam_S > 0).all()) and bool((minors_S > 0).all()),
    "Sigma is positive definite by eigenvalues and by Sylvester (Eq. 2)",
    f"eigenvalues {np.array2string(lam_S, precision=4)}, leading minors "
    f"{np.array2string(minors_S, precision=3)}, and Cholesky succeeded",
)
validate.check(
    cov_rel < 0.04,
    "x = L z reproduces the target covariance to 2% (Eq. 9)",
    f"worst relative entry error {cov_rel:.5f} over 100,000 samples",
)
validate.close(
    np.array([rho_hat]), np.array([rho_target]),
    "with the correlation matching 0.9",
    rtol=0.0, atol=1e-2,
)
validate.close(
    frac, theory,
    "and the fraction inside each k-sigma ellipse matches 1 - exp(-k^2/2) (Eq. 10)",
    rtol=0.0, atol=1e-2,
)
validate.close(
    np.sort(axes_from_inv), np.sort(axes_from_Sigma),
    "the k-sigma ellipse has semi-axes sqrt(lambda_i(Sigma)) (Eqs. 5, 9)",
    rtol=0.0, atol=1e-12,
)

---
## Notebook summary

**Five tests, one property, and two of them not usable.** On a suite of four
symmetric $3\times3$ matrices — positive definite, indefinite, semidefinite and
negative definite — the eigenvalue test, Sylvester's criterion and Cholesky
agreed on every one, accepting exactly the first. Sampling the quadratic form
over $2\times10^{4}$ random unit vectors did **not**: for the semidefinite
matrix it returned a minimum of $+1.1\times10^{-4}$, declaring it definite,
against a true minimum of $0$. And no single minor decides anything — the
negative definite matrix has leading minors $(-2, +5, -8)$.

**Cholesky is the best-behaved factorization in the course.** The
hand-written {eq}`eq-pd-cholesky-alg` reconstructed $A$ to $9\times10^{-16}$,
matched `scipy.linalg.cholesky` to $10^{-16}$, and produced an $L$ that is
*exactly* lower triangular. Each row satisfies $\|L_{i,:}\|^2 = a_{ii}$ to
$10^{-16}$, which is the growth factor being exactly 1 and is why no pivoting
is needed. It raised on all three non-definite matrices, including the
semidefinite one where $A = LL^{\top}$ exists but the algorithm must divide by
the zero it produces. Measured against `lu_factor` it was faster at every size,
though by less than the factor of 2 the flop counts predict, because both are
memory bound at $n \le 800$.

**The level set is a complete diagnosis.** For $A_2$ with eigenvalues
$(5\mp\sqrt5)/2$, the traced curve satisfied
$\mathbf{x}^{\top}\!A_2\mathbf{x} = 1$ to $9\times10^{-16}$ and its semi-axes
came out $0.850651$ and $0.525731$, matching $1/\sqrt{\lambda_i}$ to the
grid resolution — with the **longest** axis belonging to the **smallest**
eigenvalue. The indefinite form is negative on exactly half of all directions,
so its level set is a hyperbola.

**Inertia is what survives a change of coordinates.** A single congruence moved
the spectrum of $A_{\text{pd}}$ substantially while keeping every eigenvalue
positive; across 400 congruences of the four matrices the triples $(3,0,0)$,
$(2,0,1)$, $(2,1,0)$, $(0,0,3)$ never changed. For the semidefinite matrix the
nonzero eigenvalues ranged over more than two orders of magnitude while the
zero stayed zero, which is why the classification tolerance has to scale.
Cholesky itself is the congruence $L^{-1}AL^{-\top} = I$, verified to
$10^{-16}$.

**Every Gram matrix is semidefinite.** Across four shapes the worst scaled
$\lambda_{\min}$ was $-6\times10^{-17}$; the tall cases were definite and the
wide ones were not, exactly as "independent columns" predicts, with
$\operatorname{rank}(G^{\top}G) = \operatorname{rank}(G)$ throughout. The
identity $\mathbf{x}^{\top}(G^{\top}G)\mathbf{x} = \|G\mathbf{x}\|^2$ held
pointwise over $10^{4}$ vectors to a relative $10^{-16}$ — and that one line is
the entire proof.

**Cholesky generates correlated randomness.** $10^{5}$ samples drawn as
$\mathbf{x} = L\mathbf{z}$ reproduced the target covariance to $0.07\%$ and the
correlation $0.9$ to $10^{-3}$. The fractions inside the $1\sigma$ and
$2\sigma$ ellipses came out $0.3914$ and $0.8633$ against the exact
$1 - e^{-k^2/2} = 0.3935$ and $0.8647$ — a check on the whole distribution
rather than on two moments.

**Methods introduced.** `scipy.linalg.cholesky`, `cho_factor` and `cho_solve`,
the hand-written Cholesky recurrence, leading principal minors as Sylvester's
criterion, the inertia triple with a scaled zero threshold,
`scipy.linalg.solve_triangular` for congruences without forming inverses,
`np.cov`, the Mahalanobis distance $\mathbf{x}^{\top}\Sigma^{-1}\mathbf{x}$,
and `ecp.linalg.random_spd`.

## Outlook

- **The same theorem over $\mathbb{C}$, and where physics needs it.** A
  Hermitian matrix with positive eigenvalues is positive definite in exactly
  the same sense, and a density matrix in quantum mechanics is a positive
  semidefinite Hermitian matrix of trace 1 — the semidefiniteness is what makes
  its eigenvalues probabilities.
  [§3.4](hermitian-unitary-normal.ipynb) sets that up.
- **When the matrix is only nearly definite.** Estimated covariance matrices
  routinely come back with small negative eigenvalues, and the assistant
  callout above builds the standard repair. The systematic version is the
  *modified* Cholesky factorization, which computes $A + E = LDL^{\top}$ with
  $E$ as small as possible, and it is what optimisation codes use to force a
  descent direction out of an indefinite Hessian {cite}`nocedal2006`.
- **Definiteness as the hypothesis that makes iteration work.** The conjugate
  gradient method solves $A\mathbf{x} = \mathbf{b}$ using only matrix–vector
  products, and it needs $A$ positive definite for the same reason this
  notebook needed it: the quadratic form
  $\tfrac12\mathbf{x}^{\top}\!A\mathbf{x} - \mathbf{b}^{\top}\mathbf{x}$ must
  have a minimum rather than a saddle.
  [§5.4](../05-numerical/stationary-and-cg.ipynb) builds it.
- **Kernels.** A function $k(\mathbf{x}, \mathbf{y})$ is a valid kernel exactly
  when every matrix $K_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$ it produces is
  positive semidefinite — Mercer's condition, which is
  {eq}`eq-pd-gram` in disguise, since such a $k$ is always an inner product in
  some feature space. [§6.5](../06-structure/kernels-gram-matrix.ipynb) makes
  that precise.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()